
# Датасет Titanic — Разведочный анализ данных (EDA)

Источник данных: https://www.kaggle.com/competitions/titanic

В этом ноутбуке выполняется полный цикл разведочного анализа данных:

1. Загрузка данных
2. Первичный обзор
3. Обработка пропусков
4. Статистический анализ
5. Feature Engineering (создание новых признаков)
6. Кодирование категориальных признаков
7. Визуализация данных
8. Итоговые выводы

Целевая переменная:
**Survived** — выжил пассажир (1) или нет (0).


In [ ]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction import FeatureHasher


## Загрузка датасета

In [ ]:

df = pd.read_csv("train.csv")
df.head()


## Быстрый обзор данных

In [ ]:

df.head()


In [ ]:

df.tail()


In [ ]:

df.shape


In [ ]:

df.info()


In [ ]:

df.describe()


In [ ]:

df.describe(include="object")


## Проверка качества данных

In [ ]:

df.isnull().sum()


In [ ]:

df.duplicated().sum()


## Визуализация пропусков

In [ ]:

sns.heatmap(df.isnull(), cbar=False)
plt.title("Пропущенные значения")
plt.show()



### Обработка пропусков

Age → заполняем **медианой**, так как распределение может содержать выбросы.

Embarked → заполняем **модой** (самым частым значением).

Cabin → удаляем колонку, так как в ней слишком много пропусков.


In [ ]:

df['Age'].fillna(df['Age'].median(), inplace=True)


In [ ]:

df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)


In [ ]:

df.drop(columns=['Cabin'], inplace=True)


In [ ]:

df.isnull().sum()


## Расширенная статистика

In [ ]:

df[['Age','Fare']].agg(['min','max','mean','median'])


In [ ]:

df['Age'].mode()


In [ ]:

df['Fare'].quantile([0.05,0.25,0.5,0.75,0.95])


In [ ]:

variance = df['Fare'].var()
skewness = df['Fare'].skew()
kurtosis = df['Fare'].kurt()

variance, skewness, kurtosis



**Variance (дисперсия)** показывает разброс значений.

**Skewness (асимметрия)** показывает, насколько распределение смещено.
Если значение больше 0 — хвост распределения длиннее справа.

**Kurtosis (эксцесс)** показывает "остроту" распределения и наличие выбросов.


## Feature Engineering (создание новых признаков)

In [ ]:

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1


In [ ]:

df['FarePerPerson'] = df['Fare'] / df['FamilySize']


In [ ]:

df['IsChild'] = df['Age'] < 16


## Кодирование категориальных признаков

In [ ]:

df = pd.get_dummies(df, columns=['Sex','Embarked'], drop_first=True)


In [ ]:

le = LabelEncoder()
df['Ticket_encoded'] = le.fit_transform(df['Ticket'])


### Пример Feature Hashing

In [ ]:

hasher = FeatureHasher(n_features=5, input_type='string')

hashed = hasher.transform(df['Name'].astype(str))

hashed_df = pd.DataFrame(hashed.toarray())

df = pd.concat([df, hashed_df], axis=1)

df.head()


## Визуализация данных

In [ ]:

plt.hist(df['Age'], bins=30)
plt.title("Распределение возраста")
plt.xlabel("Возраст")
plt.ylabel("Количество")
plt.show()


In [ ]:

sns.kdeplot(df['Fare'], fill=True)
plt.title("Распределение стоимости билета")
plt.show()


In [ ]:

plt.scatter(df['Age'], df['Fare'])
plt.xlabel("Возраст")
plt.ylabel("Цена билета")
plt.show()


In [ ]:

sns.boxplot(x=df['Fare'])
plt.title("Выбросы в цене билета")
plt.show()


In [ ]:

sns.countplot(x='Pclass', data=df)
plt.title("Количество пассажиров по классам")
plt.show()


In [ ]:

plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True)
plt.title("Матрица корреляций")
plt.show()


In [ ]:

fig = px.scatter(df, x="Age", y="Fare", color="Survived",
                 hover_data=["Pclass","FamilySize"])
fig.show()



# Итоговые выводы

### Что удалось понять из данных

- Датасет содержит информацию о пассажирах Титаника.
- В колонке Age были пропуски, которые были заполнены медианой.
- Колонка Cabin была удалена из-за большого количества пропусков.
- Стоимость билета имеет асимметричное распределение.
- В данных присутствуют выбросы.
- Класс пассажира влияет на стоимость билета.
- Размер семьи может влиять на вероятность выживания.

### Возможные гипотезы

1. Женщины выживали чаще мужчин.
2. Пассажиры 1 класса имели больше шансов выжить.
3. Более высокая цена билета может быть связана с выживаемостью.
4. Дети могли выживать чаще взрослых.

### Что можно сделать дальше

- Построить модель **Logistic Regression**
- Попробовать **Random Forest**
- Использовать **Gradient Boosting**
- Провести отбор признаков и нормализацию данных
